In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_classification
from sklearn.tree import DecisionTreeClassifier

# ----------------------------------------------------------------
# 실험 1-1. 하드웨어 관점의 Gini vs Entropy 연산 오버헤드 실측
# ----------------------------------------------------------------
def micro_benchmark_purity():
    print("="*70)
    print("[실험 1-1] 수식적 차이에 따른 마이크로 연산 오버헤드 측정")
    print("="*70)

    # 극단적으로 큰 배열 생성 (부동소수점 및 로그 연산 스트레스 테스트)
    p = np.random.uniform(1e-5, 1-1e-5, size=5000000)

    # 1. Gini 연산 (다항식 사칙연산)
    start_gini = time.time()
    gini_val = 1 - (p**2 + (1-p)**2)
    end_gini = time.time() - start_gini
    print(f" ├ 다항식 기반 Gini 계산 시간 (500만 건)     : {end_gini:.6f} 초")

    # 2. Entropy 연산 (초월함수 로그 연산)
    start_entropy = time.time()
    entropy_val = -(p * np.log2(p) + (1-p) * np.log2(1-p))
    end_entropy = time.time() - start_entropy
    print(f" ├ 초월함수(Log2) 기반 Entropy 계산 시간      : {end_entropy:.6f} 초")

    diff_percent = (end_entropy - end_gini) / end_gini * 100
    print(f" └ [진단] CPU 연산 오버헤드 격차: Entropy가 Gini 대비 {diff_percent:.2f}% 더 느림.")

# ----------------------------------------------------------------
# 실험 1-2. 불균형 데이터셋 환경에서 엔트로피의 발산(과분할) 경향 시각화
# ----------------------------------------------------------------
def analyze_imbalance_sensitivity():
    print("\n" + "="*70)
    print("[실험 1-2] 불균형 환경에서 두 지표의 기울기(민감도) 격차 실측")
    print("="*70)

    # 클래스 불균형 99:1 데이터셋 생성
    X, y = make_classification(n_samples=5000, n_features=10, weights=[0.99, 0.01], random_state=42)

    clf_gini = DecisionTreeClassifier(criterion='gini', max_depth=10, random_state=42).fit(X, y)
    clf_entropy = DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=42).fit(X, y)

    print(f" ├ Gini 기반 트리가 생성한 총 노드 개수    : {clf_gini.tree_.node_count}개")
    print(f" ├ Entropy 기반 트리가 생성한 총 노드 개수 : {clf_entropy.tree_.node_count}개")
    print(f" └ [진단] Entropy 수식 특유의 0 부근 가파른 기울기가 소수 노이즈 분리에 집착하여 과분할을 유도함.")

if __name__ == "__main__":
    micro_benchmark_purity()
    analyze_imbalance_sensitivity()